# 14 · Melting the chocolate 🍫☕

We model a **Stefan problem** — a PDE with a *moving boundary* between solid and liquid:
* a **phase field** $\varphi$ (0 = solid, 1 = molten) carries the front in a thin diffuse layer;
* it couples to the temperature $T$ through the **latent heat** of melting:
  warming drives the transition, but turning solid into liquid *swallows* energy and holds the
  front back.

We meet it **first in 1-D** (cheap, high resolution, easy plots), then melt a **2-D cross-section
of a Toblerone**.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "anywidget", "matplotlib"], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from netgen.occ import *
from ngsolve import *
from ngsolve.meshes import Make1DMesh
from ngsolve.webgui import Draw

In [ ]:
def progress(i, n):                                    # tiny dependency-free bar
    import os
    if os.environ.get("WEBGUI_SCENE_DIR"):             # static-site build: stay silent
        return
    if (i + 1) % max(1, n // 100) == 0 or i + 1 == n:
        b = int(28 * (i + 1) / n)
        sys.stdout.write(f"\r  melting… [{'█'*b}{'·'*(28-b)}] {100*(i+1)//n:3d}%")
        sys.stdout.flush()
        if i + 1 == n:
            sys.stdout.write("\n")

## 1. The model — a tilted double well

The **phase** relaxes towards solid or liquid through a double-well reaction, across a thin
interface of width $\sim\epsilon$:
$$ \tau\,\partial_t\varphi \;=\; \epsilon^2\,\Delta\varphi
   \;+\; \underbrace{\varphi(1-\varphi)}_{\text{double well}}\bigl(\varphi-\tfrac12+m(T)\bigr),
   \qquad m(T)=\tfrac1\pi\arctan\!\bigl(\gamma\,(T-T_m)\bigr). $$

- the factor $\varphi(1-\varphi)$ vanishes at $\varphi=0$ and $\varphi=1$ — the two wells (solid, liquid);
- the **tilt** $m(T)$ decides which well wins: warm ($T>T_m$) tips it toward liquid, cold toward solid.

The temperature obeys the heat equation with a **latent** sink — each bit of melting draws heat:
$$ \partial_t T \;=\; \alpha\,\Delta T \;-\; L\,\partial_t\varphi . $$

In [ ]:
Tm, tau, alphaT, Lheat, gamma = 0.0, 1e-3, 1.0, 0.6, 4.0     # melt point, relaxation, heat-diff, latent, tilt sharpness


def reaction(phi, T):                                        # R(φ,T) = double well · tilt
    return phi * (1 - phi) * (phi - 0.5 + (1/pi)*atan(gamma*(T - Tm)))

**See the tilt.** Plot the reaction $R(\varphi)$ for several temperatures. Where $R>0$ the phase
**grows** (melts), where $R<0$ it **shrinks** (freezes); the zeros are equilibria. As $T$ rises
past $T_m$ the tilt slides the stable well from solid ($\varphi=0$) to liquid ($\varphi=1$).

In [ ]:
phs = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(6.2, 3.0))
for Tval in [-1.0, -0.3, 0.0, 0.3, 1.0]:
    mm = (1/np.pi)*np.arctan(gamma*(Tval - Tm))
    ax.plot(phs, phs*(1-phs)*(phs - 0.5 + mm), label=f"$T={Tval:+.1f}$")
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel(r"$\varphi$"); ax.set_ylabel(r"$R(\varphi,T)$")
ax.set_title("the tilted double-well reaction — warm melts, cold freezes")
ax.legend(loc="lower center", ncol=5, fontsize=8, frameon=False)
fig.tight_layout(); plt.show()

## 2. Discretising it once — and reusing it

Space first (`H1`, order 1). Both **second-order** terms go **implicitly** (factor each once);
the **local reaction** goes **explicitly** — and we write it as a **`BilinearForm(nonassemble=True)`**,
*applied* to the current state every step like a nonlinear operator, never reassembled. The hot
plate / cold base are **Dirichlet**: we invert only on the **free** dofs, so the residual update
leaves the boundary values exactly in place.

In [ ]:
def run_melt(mesh, gfphi, gfT, dt, eps2, nsteps, nframes, profile=False):
    """March the phase-field Stefan problem; return the φ-animation (and, optionally, 1-D profiles)."""
    fes = gfphi.space
    phi, psi = fes.TnT()
    react = BilinearForm(fes, nonassemble=True)                # explicit reaction R(φ,T), applied each step
    react += reaction(phi, gfT) * psi * dx
    M = BilinearForm(phi*psi*dx).Assemble()
    K = BilinearForm(grad(phi)*grad(psi)*dx).Assemble()
    Aphi = M.mat.CreateMatrix(); Aphi.AsVector().data = tau*M.mat.AsVector() + dt*eps2*K.mat.AsVector()
    invphi = Aphi.Inverse(fes.FreeDofs(), inverse="sparsecholesky")     # free dofs only → Dirichlet kept
    AT = M.mat.CreateMatrix(); AT.AsVector().data = M.mat.AsVector() + dt*alphaT*K.mat.AsVector()
    invT = AT.Inverse(fes.FreeDofs(), inverse="sparsecholesky")
    res, phi_old = gfphi.vec.CreateVector(), gfphi.vec.CreateVector()
    anim = GridFunction(fes, multidim=0); anim.AddMultiDimComponent(gfphi.vec)
    verts = [mesh(*v.point) for v in mesh.vertices] if profile else None
    snaps = [(np.array([gfphi(p) for p in verts]), np.array([gfT(p) for p in verts]))] if profile else []
    snap = max(1, nsteps // nframes)
    with TaskManager():
        for step in range(nsteps):
            phi_old.data = gfphi.vec
            react.Apply(gfphi.vec, res)                                  # explicit reaction
            gfphi.vec.data += invphi*(tau*M.mat*gfphi.vec + dt*res - Aphi*gfphi.vec)   # + implicit ε²-diffusion
            dphi = gfphi.vec - phi_old
            gfT.vec.data += invT*(M.mat*(gfT.vec - Lheat*dphi) - AT*gfT.vec)           # latent-heat sink
            if (step + 1) % snap == 0:
                anim.AddMultiDimComponent(gfphi.vec)
                if profile:
                    snaps.append((np.array([gfphi(p) for p in verts]), np.array([gfT(p) for p in verts])))
            progress(step, nsteps)
    return (anim, verts, snaps) if profile else anim


def set_on(gf, region, value):                                          # set a boundary's dofs, leave the interior
    d = gf.space.GetDofs(gf.space.mesh.Boundaries(region))
    for i in range(len(d)):
        if d[i]:
            gf.vec[i] = value

## 3. First in 1-D — a heated bar

A bar $[0,4]$ with the **hot plate** at the left ($\varphi=1$, $T=3$); everything else starts
**cold and solid** ($T=-1$, $\varphi=0$). High resolution is cheap in 1-D, so we just plot the
**profiles** as the melt front marches in.

In [ ]:
mesh1d = Make1DMesh(200, mapping=lambda t: 4.0*t)            # the bar, left end = "left" = hot plate
fes1d  = H1(mesh1d, order=1, dirichlet="left")
gfphi  = GridFunction(fes1d); gfphi.Set(0.0); set_on(gfphi, "left", 1.0)
gfT    = GridFunction(fes1d); gfT.Set(-1.0);  set_on(gfT,   "left", 3.0)

anim1d, verts, snaps = run_melt(mesh1d, gfphi, gfT, dt=2e-4, eps2=4e-4,
                                nsteps=4000, nframes=4, profile=True)
print(f"\nmolten fraction: {Integrate(gfphi, mesh1d)/4:.0%} of the bar")

In [ ]:
xs = np.array([v.point[0] for v in mesh1d.vertices]); order = np.argsort(xs)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3))
for k, (ph, T) in enumerate(snaps):
    c = plt.cm.copper(0.2 + 0.8*k/(len(snaps)-1))
    a1.plot(xs[order], ph[order], color=c, label=f"frame {k}")
    a2.plot(xs[order], T[order], color=c)
a1.axhline(0.5, color="gray", lw=0.6, ls="--"); a1.set_title("phase φ (0.5 = the front)")
a2.axhline(Tm, color="gray", lw=0.6, ls="--"); a2.set_title("temperature T ($T_m$ dashed)")
for a in (a1, a2): a.set_xlabel("x")
a1.legend(fontsize=8, frameon=False); fig.tight_layout(); plt.show()

The front advances quickly at first, then **slows** — it has to wait for heat to diffuse deeper,
and every bit of melting *spends* latent heat ($L\,\partial_t\varphi$), which is exactly what holds
a real melt front back.

## 4. The Toblerone — a 2-D cross-section

Now the iconic shape: a **cross-section through a Toblerone**, the row of triangular teeth, resting
on a **cold plate** (the bottom edge). The tooth surfaces are **warm** ($T=2$, molten skin), the
**base** stays cold ($T=-1$, solid) — so the chocolate melts from the surface inward while the base
anchors it. *(The full 3-D bar, clipped through the middle, is the natural next step — left for a
rainy day.)*

In [ ]:
N, W, H, vh = 5, 8.0, 1.8, 0.4                               # 5 teeth, width, peak height, valley height
# Trace the outline starting at the bottom and going counter-clockwise (so OCC keeps the *inside*).
# The edge teeth taper all the way to the bottom corners — no vertical cut at the ends.
wp = WorkPlane().MoveTo(0, 0).LineTo(W, 0)                   # the bottom (the cold plate)
for i in range(N - 1, -1, -1):                              # up the right, across the teeth right → left
    wp.LineTo((i + 0.5)*W/N, H)                              # peak of tooth i
    wp.LineTo(i*W/N, 0.0 if i == 0 else vh)                  # next valley (height vh); the last runs to the corner
tob = wp.Close().Face()
tob.edges.name = "warm"                                     # tooth surfaces: warm
tob.edges.Min(Y).name = "base"                              # bottom edge: the cold plate
mesh = Mesh(OCCGeometry(tob, dim=2).GenerateMesh(maxh=0.16))

fes  = H1(mesh, order=1, dirichlet="warm|base")
gfphi = GridFunction(fes); gfphi.Set(0.0); set_on(gfphi, "warm", 1.0)    # molten skin, solid interior + base
gfT   = GridFunction(fes); gfT.Set(-1.0);  set_on(gfT,   "warm", 2.0)    # warm skin, cold interior + base

anim = run_melt(mesh, gfphi, gfT, dt=5e-4, eps2=2e-3, nsteps=2000, nframes=16)
print(f"\nmolten fraction: {Integrate(gfphi, mesh)/Integrate(CF(1), mesh):.0%} of the bar")
Draw(anim, mesh, "melting Toblerone — φ (0 = solid, 1 = molten)",
     interpolate_multidim=True, animate=True, min=0, max=1, autoscale=False)

The teeth melt from their warm faces inward; the cold base keeps a solid foot. Same model, same
`run_melt` — only the **geometry** and the **boundary roles** changed.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):
    _prev = ("13-thermal-plume-hdg", "13 · A puffing thermal plume — HDiv-HDG & HDG 🔥🌀")
    _next = ("15-outlook-unfitted", "15 · The Grand Expedition — deforming domains (unfitted FEM) 🫧")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))